In [ ]:
import numpy as np
import numpy.ma as ma
import logging
import os
import warnings
import yaml

# https://github.com/DUNE/ndlar_flow/blob/3944a930d34fb7612bd4305162e3f6a6ce9dbb2e/yamls/proto_nd_flow/resources/GeometryData.yaml
prefix = '/home/yousen/Public/ndlar_shared/data/diagnostics/ndlar_flow'
params = {
        'det_geometry_file': f'{prefix}/data/proto_nd_flow/2x2.yaml',
      'crs_geometry_files': [f'{prefix}/data/proto_nd_flow/layouts_v5/multi_tile_layout-2.3.16_mod0_swap_T8T4T7.yaml',
                       f'{prefix}/data/proto_nd_flow/layouts_v5/multi_tile_layout-2.3.16_mod1_noswap.yaml',
                       f'{prefix}/data/proto_nd_flow/layouts_v5/multi_tile_layout-2.5.16_mod2_swap_T7T8.yaml',
                       f'{prefix}/data/proto_nd_flow/layouts_v5/multi_tile_layout-2.3.16_mod3_swap_T5T8_T9T10.yaml'],
      'crs_geometry_to_module': [0,1,2,3]
}
class Geometry():
    '''
    '''
    class_version = '0.2.0'

    default_n_io_channels_per_tile = 4

    def __init__(self, **params):


        self.n_io_channels_per_tile = params.get('n_io_channels_per_tile', self.default_n_io_channels_per_tile)
        self.crs_geometry_files = params.get('crs_geometry_files')
        self.det_geometry_file = params.get('det_geometry_file')
        self.crs_geometry_to_module = params.get('crs_geometry_to_module')

        
        self._tile_id = {}
        self._load_charge_geometry()


    def _load_charge_geometry(self):
        geometry_yamls = []
        for crs_geometry_file in self.crs_geometry_files:
            with open(crs_geometry_file) as gf:
                geometry_yamls.append(yaml.load(gf, Loader=yaml.FullLoader))
                if 'multitile_layout_version' not in geometry_yamls[-1].keys():
                    raise RuntimeError('Only multi-tile geometry configurations are accepted')

        with open(self.det_geometry_file) as dgf:
            det_geometry_yaml = yaml.load(dgf, Loader=yaml.FullLoader)

        module_to_io_groups = det_geometry_yaml['module_to_io_groups']

        tile_geometry = {}

        ## warning, this is assuming same number of tiles in all modules for now
        tiles = np.arange(1,len(geometry_yamls[0]['tile_chip_to_io'])*len(det_geometry_yaml['module_to_io_groups'])+1)
        io_groups = [
            geometry_yamls[self.crs_geometry_to_module[mod-1]]['tile_chip_to_io'][tile][chip] // 1000 + (mod-1)*4
            for mod in module_to_io_groups
            for tile in geometry_yamls[self.crs_geometry_to_module[mod-1]]['tile_chip_to_io']
            for chip in geometry_yamls[self.crs_geometry_to_module[mod-1]]['tile_chip_to_io'][tile]
        ]
        io_channels = [
            geometry_yamls[self.crs_geometry_to_module[mod-1]]['tile_chip_to_io'][tile][chip] % 1000
            for mod in module_to_io_groups
            for tile in geometry_yamls[self.crs_geometry_to_module[mod-1]]['tile_chip_to_io']
            for chip in geometry_yamls[self.crs_geometry_to_module[mod-1]]['tile_chip_to_io'][tile]
        ]
        chip_ids = [
            chip_channel // 1000
            for mod in module_to_io_groups
            for chip_channel in geometry_yamls[self.crs_geometry_to_module[mod-1]]['chip_channel_to_position']
        ]
        channel_ids = [
            chip_channel % 1000
            for mod in module_to_io_groups
            for chip_channel in geometry_yamls[self.crs_geometry_to_module[mod-1]]['chip_channel_to_position']
        ]
    
        tile_min_max = [(min(v), len(module_to_io_groups)*max(v)) for v in (io_groups, io_channels)]
    
        anode_min_max = [(min(tiles), len(module_to_io_groups)*max(tiles))]


        mod_centers = det_geometry_yaml['tpc_offsets']
        n_modules = len(det_geometry_yaml['module_to_io_groups'])
        n_tiles = sum(len(j) for i in det_geometry_yaml['tile_map'] for j in i)
        # DOUBLE WARNING!: I'm doing a terrible thing and hardcoding things based on
        #                  the first geometry file option in the list...
        #                  Please, fix me! (move into loop below)
        tile_or = geometry_yamls[0]['tile_orientations']
        tile_pos = geometry_yamls[0]['tile_positions']

        # Loop through modules
        for module_id in module_to_io_groups:
            geometry_yaml = geometry_yamls[self.crs_geometry_to_module[module_id-1]]
            chip_channel_to_position = geometry_yaml['chip_channel_to_position']
            tile_orientations = geometry_yaml['tile_orientations']
            tile_positions = geometry_yaml['tile_positions']
            tile_chip_to_io = geometry_yaml['tile_chip_to_io']

            for tile in tile_chip_to_io:
                tile_orientation = tile_orientations[tile]

                for chip in tile_chip_to_io[tile]:
                    io_group_io_channel = tile_chip_to_io[tile][chip]
                    io_group = io_group_io_channel//1000 + (module_id-1)*len(det_geometry_yaml['module_to_io_groups'][module_id])
                    io_channel = io_group_io_channel % 1000
                    
                    tile_id = tile+(module_id-1)*len(tile_chip_to_io)
                    self._tile_id[(io_group, io_channel)] = tile_id
                        
        for k, v in self._tile_id.items():
            tid = 1+(k[0]-1)*8+(k[1]-1)//4
            print(k, v, tid, tid==v)

In [ ]:
Geometry(**params)